# Bluetooth Low Energy — advertisements, GATT, notifications

A central scans advertisements, connects to peripherals and reads, writes and subscribes to characteristics. The virtual radio below behaves like the real one; switch to `o.UseNative()` to use your Bluetooth adapter.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.17.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.17.0-preview.1"

## Decode an advertisement by hand
Advertising data is a list of length–type–value structures; an iBeacon is Apple manufacturer data.

In [ ]:
using IoTCom.Net.Transport.Ble;

var raw = Convert.FromHexString("020106" + "03030D18" + "040948524D" + "1AFF4C000215E2C56DB5DFFB48D2B060D0F5A71096E000070001C5");
var parsed = AdvertisingData.Parse("demo", raw, rssi: -68);
Console.WriteLine($"{parsed.Name} · services {string.Join(",", parsed.Services.Select(BleUuid.Name))} · iBeacon {parsed.IBeacon} · ≈{parsed.EstimatedDistance} m");
foreach (var f in AdvertisingData.Describe(raw)) Console.WriteLine($"  {f.Name,-5} {f.Value}");

## Scan a (virtual) room

In [ ]:
var room = new VirtualBleNetwork();
room.AddHeartRateStrap();
room.AddEnvironmentSensor();
room.AddBeacon();
room.AddSmartPlug();
var physics = new VirtualBleSimulator(room);
var ble = BleCentral.Create(o => { o.UseVirtual(room); o.ReadOnly = true; });
await ble.ConnectAsync();
foreach (var ad in await ble.ScanAsync(TimeSpan.FromMilliseconds(600)))
    Console.WriteLine($"{ad.Name,-18} {ad.Rssi,4} dBm  {string.Join(", ", ad.Services.Select(BleUuid.Name))}{(ad.IBeacon is { } b ? $" iBeacon {b.Major}/{b.Minor}" : "")}");

## Connect, read, subscribe

In [ ]:
var strap = await ble.OpenAsync("C4:7C:8D:6A:21:0F");
foreach (var svc in strap.Services) Console.WriteLine($"{svc.Name}: {string.Join(", ", svc.Characteristics.Select(c => $"{c.Name} ({c.Properties})"))}");
Console.WriteLine($"battery {GattValue.Describe(BleUuid.Parse("2a19"), await strap.ReadAsync("2a19"))}");
using var beatWindow = new CancellationTokenSource(TimeSpan.FromSeconds(3));
_ = Task.Run(async () => { while (!beatWindow.IsCancellationRequested) { physics.Step(); await Task.Delay(250); } });
try { await foreach (var v in strap.SubscribeAsync(BleUuid.FromShort(0x2A37), beatWindow.Token)) Console.WriteLine(GattValue.Describe(BleUuid.FromShort(0x2A37), v)); }
catch (OperationCanceledException) { }

## Writes are refused in read-only mode

In [ ]:
var plug = await ble.OpenAsync("D0:8E:3A:55:10:C2");
try { await plug.WriteAsync(VirtualBleNetwork.SmartPlugRelay, new byte[] { 1 }); }
catch (IoTCom.Net.ReadOnlyModeException e) { Console.WriteLine("read-only: " + e.Message); }
await ble.DisposeAsync();

## Going further
`iotcom ble scan` (real radio) or `--sim`, `iotcom ble watch --sim C4:7C:8D:6A:21:0F 2a37`, the Gallery's *Nearby Bluetooth devices* and the BleHeartRate sample. See `docs/en/protocols/ble.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*